# 03 -- PyMuPDF

## Objectifs pedagogiques

A la fin de ce notebook, vous saurez :

- ouvrir et manipuler un PDF avec PyMuPDF (`fitz`) ;
- extraire texte, images et metadonnees ;
- rendre une page en image (PNG) ;
- annoter et dessiner sur un PDF ;
- rechercher du texte (find) ;
- extraire en JSON structure ;
- decouper, merger, rotater, inserer des pages ;
- OCR avec tesseract (introduction).

## Prerequis -- ce que vous devez maitriser

- fonctions typees, pathlib ;
- Pillow (pour manipulation images extraites).

Ce que ce notebook **ne couvrira pas** :

- l'ecriture de templates HTML -> PDF (voir `05_Weasyprint.ipynb`) ;
- les Platypus flowables (voir `04_Reportlab_platypus.ipynb`).

## Plan

1. Installation
2. Ouvrir un PDF
3. Extraire texte
4. Metadonnees
5. Rendu en image
6. Extraire images
7. Recherche de texte
8. Annotations et dessin
9. Insertion de pages
10. JSON structure
11. OCR (intro)
12. Synthese
13. Exercices

---

## 1. Installation

In [ ]:
# pip install pymupdf
import fitz  # PyMuPDF
print(fitz.version)

---

## 2. Ouvrir un PDF

In [ ]:
import tempfile, pathlib
from reportlab.pdfgen import canvas
tmp = pathlib.Path(tempfile.gettempdir())
path = tmp / 'demo.pdf'
cc = canvas.Canvas(str(path))
cc.drawString(100, 750, 'Hello PyMuPDF')
cc.showPage()
cc.drawString(100, 750, 'Page deux')
cc.save()

doc = fitz.open(str(path))
print('Pages :', doc.page_count)

---

## 3. Extraire texte

In [ ]:
page = doc[0]
print(page.get_text())

In [ ]:
# Different modes d'extraction
print(page.get_text('text')[:60])
print('---')
print(page.get_text('html')[:200])

---

## 4. Metadonnees

In [ ]:
doc.metadata

In [ ]:
doc.set_metadata({'title': 'Demo', 'author': 'Alice'})
doc.metadata

---

## 5. Rendu en image

In [ ]:
pix = page.get_pixmap(dpi=150)
out = tmp / 'page1.png'
pix.save(str(out))
print('Image :', pix.width, 'x', pix.height)

In [ ]:
# Haute resolution
pix2 = page.get_pixmap(matrix=fitz.Matrix(2, 2))  # 2x zoom
pix2.width, pix2.height

---

## 6. Extraire images

In [ ]:
# Lister les images d'une page
for img_info in page.get_images(full=True):
    xref = img_info[0]
    pix = fitz.Pixmap(doc, xref)
    print('xref', xref, 'w=', pix.width, 'h=', pix.height)
print('Pas d\'image dans ce demo')

---

## 7. Recherche de texte

In [ ]:
hits = page.search_for('Hello')
for rect in hits:
    print('trouve a :', rect)

---

## 8. Annotations et dessin

In [ ]:
# Surligner
if hits:
    annot = page.add_highlight_annot(hits[0])
    annot.update()
print('Highlight ajoute')

In [ ]:
# Dessiner une ligne
page.draw_line((50, 100), (200, 100), color=(1, 0, 0), width=2)
print('Ligne dessinee')

In [ ]:
# Inserer du texte
page.insert_text((100, 500), 'Texte ajoute', fontsize=14, color=(0, 0, 1))
print('Texte ajoute')

In [ ]:
# Sauvegarder
edited = tmp / 'edited.pdf'
doc.save(str(edited))
print('Sauvegarde dans', edited)

---

## 9. Insertion de pages

In [ ]:
doc2 = fitz.open(str(path))
new_doc = fitz.open()
new_doc.insert_pdf(doc2, from_page=0, to_page=0)  # copie la 1re page
print('Nouveau doc :', new_doc.page_count, 'pages')

In [ ]:
# Inserer une page blanche
new_doc.insert_page(-1, text='Page ajoutee', fontsize=20)
new_doc.page_count

---

## 10. JSON structure

In [ ]:
# Structure complete
import json
json_text = page.get_text('dict')
print(type(json_text).__name__)
print(list(json_text.keys()))
print(len(json_text['blocks']), 'blocs')

---

## 11. OCR (intro)

PyMuPDF integre l'OCR via Tesseract.

```bash
pip install pymupdf pytesseract
sudo apt install tesseract-ocr tesseract-ocr-fra
```

```python
# page.get_textpage_ocr(language='fra', dpi=300)
# text = page.get_text('text', textpage=tp)
```

Utile pour extraire du texte de PDF scannes (images).

---

## Synthese

| Operation | PyMuPDF |
|---|---|
| Ouvrir | `fitz.open(path)` |
| Pages | `doc[n]` ou `doc.page_count` |
| Texte | `page.get_text()` / `'html'` / `'dict'` |
| Rendu | `page.get_pixmap(dpi=150)` |
| Rechercher | `page.search_for('mot')` |
| Highlight | `page.add_highlight_annot(rect)` |
| Dessiner | `page.draw_line/rect/circle` |
| Texte insere | `page.insert_text((x, y), s)` |
| Fusion | `new_doc.insert_pdf(src, from_page, to_page)` |
| OCR | `page.get_textpage_ocr(language='fra')` |

### Regles a retenir

- **PyMuPDF est le plus rapide et le plus complet** pour manipuler des PDF.
- **Licence AGPL** : impose l'open-source des distributions ou licence commerciale.
- **Coordonnees** : origine en haut-gauche (contraire du PDF natif).
- **Toujours `doc.close()`** ou utiliser `with`.

---

## Exercices

### Exercice 1 (facile) -- Nb pages

Fonction `page_count(path)` qui retourne le nombre de pages.

In [ ]:
import fitz

# TODO

<details>
<summary>Voir la correction</summary>

```python
import fitz

def page_count(path: str) -> int:
    with fitz.open(path) as doc:
        return doc.page_count
```

</details>

### Exercice 2 (facile) -- PDF vers PNG

Fonction `pdf_to_png(pdf_path, page_num, out_path, dpi)`.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import fitz

def pdf_to_png(pdf_path: str, page_num: int, out_path: str, dpi: int = 150) -> None:
    with fitz.open(pdf_path) as doc:
        pix = doc[page_num].get_pixmap(dpi=dpi)
        pix.save(out_path)
```

</details>

### Exercice 3 (moyen) -- Recherche et highlight

Fonction `highlight_word(pdf_in, pdf_out, word)` qui surligne toutes les occurrences du mot.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import fitz

def highlight_word(pdf_in: str, pdf_out: str, word: str) -> int:
    count = 0
    with fitz.open(pdf_in) as doc:
        for page in doc:
            for rect in page.search_for(word):
                page.add_highlight_annot(rect).update()
                count += 1
        doc.save(pdf_out)
    return count
```

</details>

### Exercice 4 (moyen) -- Extraction texte complet

Fonction `full_text(path) -> str` qui concatene le texte de toutes les pages.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import fitz

def full_text(path: str) -> str:
    with fitz.open(path) as doc:
        return '\n'.join(p.get_text() for p in doc)
```

</details>

### Exercice 5 (difficile) -- Ajouter un watermark sur toutes les pages

Fonction `add_watermark(pdf_in, pdf_out, text)` qui ajoute un texte en diagonal semi-transparent au centre de chaque page.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import fitz

def add_watermark(pdf_in: str, pdf_out: str, text: str) -> None:
    with fitz.open(pdf_in) as doc:
        for page in doc:
            w, h = page.rect.width, page.rect.height
            page.insert_text(
                (w / 2 - 100, h / 2),
                text,
                fontsize=40,
                rotate=45,
                color=(0.7, 0.7, 0.7),
            )
        doc.save(pdf_out)
```

</details>

---

## Ressources externes

- [PyMuPDF documentation](https://pymupdf.readthedocs.io/)
- [GitHub](https://github.com/pymupdf/PyMuPDF)
- [Tesseract OCR](https://github.com/tesseract-ocr/tesseract)